# QF 627 Programming and Computational Finance
## Group Take-Home Assessment | PCA books

## <a id = "top">Quick Nav</a>

* [Step 0. Libraries, the desk file and its extension](#s0)
* [Part 1. Build the candidate books](#part1)
    * [Step 1. The desk's limits and the pieces every book shares](#p1s1)
    * [Step 2. PCA of the decision day's window](#p1s2)
    * [Step 3. Family A: the max-variance book](#p1s3)
    * [Step 4. Family B: residuals, the mean-reversion fit and the s-score](#p1s4)
    * [Step 5. Family B: replay the trading rule and turn it into weights](#p1s5)
    * [Step 6. Four versions from two functions, and the yardsticks](#p1s6)
    * [Step 7. Check every candidate on one decision day](#p1s7)
* [Part 2. Backtest the candidates](#part2)
    * [Step 8. The desk engine and the scorecard](#p2s8)
    * [Step 9. Walk forward, 3 January 2017 to 5 November 2026](#p2s9)
    * [Step 10. Research window and holdout scorecards](#p2s10)
    * [Step 11. Charts](#p2s11)
    * [Step 12. Is the research Sharpe ratio luck? t-statistic and deflated Sharpe ratio](#p2s12)
    * [Step 13. Placebo](#p2s13)
    * [Step 14. The ten-day lens of the holdout](#p2s14)
* [The days seen before the deadline, 9 to 16 November](#live)
    * [Step 15. What the books did on the six days](#lives15)
    * [Step 16. Weights at every close from 6 to 16 November](#lives16)
    * [Step 17. How the four versions relate, and the names held at the 16 November close](#lives17)
* [Self-critique of the books](#part4)

## <a id = "s0">Step 0.</a> Libraries, the desk file and its extension [back to top](#top)

In [1]:
# Activate the libraries
import time
import warnings

from functools import partial

import numpy as np
import pandas as pd

from scipy.stats import kurtosis
from scipy.stats import norm
from scipy.stats import skew

from lets_plot import *

LetsPlot.setup_html()

warnings.filterwarnings("ignore")

SEED = 627

np.set_printoptions(precision = 4, suppress = True)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

In [2]:
# Load the frozen desk file and set the desk's constants (Problem Set 8)
prices_frozen =\
(
    pd
    .read_csv("prices.csv",
              index_col = 0,
              parse_dates = True
             )
)

STOCKS = ["NVDA", "GOOGL", "AMZN", "AAPL", "MSFT", "JPM", "JNJ", "CAT", "XOM", "WMT", "META", "KO"]
TICKERS = STOCKS + ["SPY"]

CAPITAL = 1_000_000
COST = 0.0010
STOCK_CAP = 0.25
VOL_LIMIT = 0.20
VOL_TARGET = 0.195             # target volatility, below the 20% limit
RISK_DAYS = 63
ESTIMATION_DAYS = 252

FIRST_DECISION = "2017-01-03"
RESEARCH_START = "2017-01-04"
RESEARCH_END = "2024-12-31"
HOLDOUT_START = "2025-01-02"
HOLDOUT_END = "2026-11-06"
WALK_LAST_DECISION = "2026-11-05"
LIVE_FIRST_DECISION = "2026-11-06"
LIVE_START = "2026-11-09"

RETURN_TOL = 1e-6              # largest gap allowed between fresh and frozen returns on the overlap
CALL_LIMIT_SECONDS = 30        # longest time allowed for one call of a book, in seconds (R11)

print("frozen desk file :", prices_frozen.shape)
print("first day        :", f"{prices_frozen.index[0]:%Y-%m-%d}")
print("last day         :", f"{prices_frozen.index[-1]:%Y-%m-%d}")

frozen desk file : (2722, 13)
first day        : 2016-01-04
last day         : 2026-10-29


> The cell checks the file (13 tickers in order, unique increasing dates, no missing, non-finite or non-positive price) and raises an error on any problem. It removes nothing.

In [3]:
# Check the price file and raise an error on any problem
def audit_prices(frame):
    values = frame[TICKERS].to_numpy(dtype = float)
    returns = frame[TICKERS].pct_change(fill_method = None)
    summary = pd.Series({"rows": len(frame),
                         "columns": frame.shape[1],
                         "duplicate dates": int(frame.index.duplicated().sum()),
                         "increasing dates": bool(frame.index.is_monotonic_increasing),
                         "weekend rows": int((frame.index.dayofweek >= 5).sum()),
                         "missing values": int(frame[TICKERS].isna().sum().sum()),
                         "non-finite values": int((~np.isfinite(values)).sum()),
                         "non-positive values": int((values <= 0).sum())})
    by_ticker = pd.DataFrame({"days without a move": returns.eq(0).sum(),
                              "largest daily move": returns.abs().max()})
    return summary, by_ticker


def validate_prices(frame):
    if frame.columns.tolist() != TICKERS:
        raise ValueError(f"Expected the columns {TICKERS}, found {frame.columns.tolist()}")
    summary, _ = audit_prices(frame)
    if summary["duplicate dates"] or not summary["increasing dates"]:
        raise ValueError("Dates must be unique and increasing")
    if summary["missing values"] or summary["non-finite values"] or summary["non-positive values"]:
        raise ValueError("Adjusted closes must be present, finite and positive")


validate_prices(prices_frozen)
audit_summary, audit_by_ticker = audit_prices(prices_frozen)
display(audit_summary.to_frame("frozen file"))
audit_by_ticker.style.format({"days without a move": "{:,.0f}", "largest daily move": "{:.2%}"})

,frozen file
rows,2722
columns,13
duplicate dates,0
increasing dates,True
weekend rows,0
missing values,0
non-finite values,0
non-positive values,0


,days without a move,largest daily move
NVDA,3,29.81%
GOOGL,3,12.64%
AMZN,4,15.32%
AAPL,6,15.33%
MSFT,13,15.51%
JPM,10,18.01%
JNJ,7,10.04%
CAT,1,14.28%
XOM,7,12.69%
WMT,7,11.71%


> R12: chain fresh **returns** onto the last frozen close, $P_t = P_{t-1}(1 + r_t)$, drop a live New York row, and stop with an error if the overlap disagrees with the frozen file.

In [4]:
from datetime import datetime
from datetime import time as clock_time        
from zoneinfo import ZoneInfo

fresh = pd.read_csv("extended_prices.csv", index_col=["Date"])

In [5]:
fresh_index = pd.DatetimeIndex(fresh.index)
if fresh_index.tz is not None:
    fresh_index = fresh_index.tz_convert("America/New_York").tz_localize(None)
fresh.index = pd.DatetimeIndex(fresh_index.normalize(), name = "date")
fresh = fresh.sort_index()

# drop today's row if New York has not closed yet
now_ny = datetime.now(ZoneInfo("America/New_York"))
if fresh.index[-1].date() == now_ny.date() and now_ny.time() < clock_time(16, 5):
    fresh = fresh.iloc[:-1]

def extend_prices(prices_frozen, fresh):
    # chain the fresh daily returns onto the last frozen close (R12)
    fresh_returns = fresh.pct_change(fill_method = None)
    frozen_returns = prices_frozen.pct_change(fill_method = None)
    
    last = prices_frozen.index[-1]
    overlap = frozen_returns.index.intersection(fresh_returns.dropna(how = "any").index)
    gap = (fresh_returns.loc[overlap] - frozen_returns.loc[overlap]).abs().max().max()
    new = fresh_returns.loc[fresh_returns.index > last].dropna(how = "any")
    chained = (1 + new).cumprod().mul(prices_frozen.iloc[-1])
    
    return pd.concat([prices_frozen, chained]).rename_axis("date"), gap, len(overlap)

In [6]:
prices, gap, overlap_days = extend_prices(prices_frozen[TICKERS], fresh)

# Checks (in case a different extended_prices.csv is used)
if fresh.empty or fresh.index.duplicated().any():
    raise ValueError("The extended prices file is empty or has duplicate dates")
if overlap_days < 3:
    raise ValueError("Fewer than three complete overlapping return days")
if gap > RETURN_TOL:
    raise ValueError(f"The fresh and frozen returns disagree on the overlap by {gap:.3e}")

validate_prices(prices)
print("overlap days compared:", overlap_days, "| largest gap between fresh and frozen returns:", f"{gap:.2e}", "| tolerance:", f"{RETURN_TOL:.0e}")
print("rows:", len(prices_frozen), "->", len(prices), "| last close:", f"{prices.index[-1]:%Y-%m-%d}")

if pd.Timestamp(HOLDOUT_END) not in prices.index:
    raise ValueError("The extended file does not reach the 6 November close; the holdout would be cut short.")

overlap days compared: 33 | largest gap between fresh and frozen returns: 8.29e-08 | tolerance: 1e-06
rows: 2722 -> 2734 | last close: 2026-11-16


---

## <font color = "purple"> <a id = "part1">Part 1.</a> Build the candidate books [back to top](#top)

> **Family A, max-variance book.** Takes PC1, the combination of stocks that moves the most, keeps its positive weights, caps each stock at 25% and scales the book to the volatility limit. Parameter: `matrix`, the covariance (`cov`) or the correlation (`corr`) matrix.

> **Family B, PCA stat-arb book.** Splits each stock's return into a part the PCA factors explain and a leftover, then scores how stretched the leftover is (the s-score). Oversold names go to double weight and overbought names to zero, around equal weight, since the desk allows no short sales. Parameter: `k_rule`, how many factors to keep (`variance` or `noise`).

### <a id = "p1s1">Step 1.</a> The desk's limits and the pieces every book shares [back to top](#top)

> Every book ends with `finish_book`: add SPY, scale to a 19.5% volatility target (below the 20% limit) and return the 13 weights. `check_limits` is R5.

In [7]:
# Cap, volatility scaler and limit checks shared by every book (R5, R7)
def cap_weights(w, cap = STOCK_CAP):
    # cap every weight and share the excess among the other held names in proportion to their weights,
    # until none is above the cap; works row by row on a 2-D array too
    vector = np.ndim(w) == 1
    w = np.atleast_2d(np.asarray(w, dtype = float)).copy()
    for _ in range(w.shape[1]):
        over = w > cap + 1e-12
        if not over.any():
            break
        excess = np.where(over, w - cap, 0.0).sum(axis = 1, keepdims = True)
        w = np.where(over, cap, w)
        room = (w > 0) & (w < cap)
        room_total = np.where(room, w, 0.0).sum(axis = 1, keepdims = True)
        w = np.where(room & (room_total > 0), w + excess * w / np.where(room_total > 0, room_total, 1.0), w)
    return w[0] if vector else w


def vol_scaler(weights, returns_13, target = VOL_TARGET):
    # ex-ante volatility on the sample covariance of the last 63 returns of all 13 tickers
    sigma_63 = np.cov(returns_13[-RISK_DAYS:].T)
    vol = np.sqrt(252 * weights @ sigma_63 @ weights)
    return weights * min(1.0, target / vol) if vol > 0 else weights


def recent_returns(prices, days, tickers = TICKERS):
    # simple daily returns of the last `days` days
    closes = prices[tickers].values[-(days + 1):]
    return closes[1:] / closes[:-1] - 1


def finish_book(stock_weights, prices, spy = 0.0, target = VOL_TARGET):
    # add SPY's weight, scale down to the volatility target and name the thirteen weights
    weights = vol_scaler(np.append(stock_weights, spy), recent_returns(prices, RISK_DAYS), target)
    return pd.Series(weights, index = TICKERS)


# check the five limits of R5, with 1e-9 on the weights and 1e-6 on the volatility
def ex_ante_vol(weights, prices):
    sigma_63 = prices[TICKERS].iloc[-(RISK_DAYS + 1):].pct_change().iloc[1:].cov().values
    w = weights.reindex(TICKERS).values
    return np.sqrt(252 * w @ sigma_63 @ w)


def check_limits(weights, prices):
    w = pd.Series(weights, dtype = float)
    if not set(w.index).issubset(TICKERS) or not np.isfinite(w.values).all():
        return False
    w = w.reindex(TICKERS).fillna(0.0)
    return bool((w >= -1e-9).all()
                and w.sum() <= 1 + 1e-9
                and (w[STOCKS] <= STOCK_CAP + 1e-9).all()
                and w["SPY"] <= 1 + 1e-9
                and ex_ante_vol(w, prices) <= VOL_LIMIT + 1e-6)

### <a id = "p1s2">Step 2.</a> PCA of the decision day's window [back to top](#top)

> One eigen-decomposition for both families, sorted from the largest eigenvalue, with the sign rule of R3.

In [8]:
# Eigen-decomposition of the covariance or correlation matrix (Week 8 notebook)
def eigen_pairs(block, matrix = "corr"):
    # eigenvalues from the largest to the smallest, with their eigenvectors in the columns,
    # each turned so that its elements sum to a positive number (R3)
    table = np.cov(block.T) if matrix == "cov" else np.corrcoef(block.T)
    lam, vec = np.linalg.eigh(table)
    order = np.argsort(lam)[::-1]
    lam, vec = lam[order], vec[:, order]
    return lam, vec * np.sign(vec.sum(axis = 0))

### <a id = "p1s3">Step 3.</a> Family A: the max-variance book [back to top](#top)

> PC1's positive loadings, rescaled to sum to 1, capped at 25% and scaled to the volatility target. With `corr`, each loading is divided by the stock's own volatility.

In [9]:
def maxvar_weights(block, matrix = "cov", n_pc = 1):
    # long-only PC1 weights of a 252-day block of the 12 stocks' returns, capped at 25%;
    # sums to 1 unless fewer than four are positive
    lam, vec = eigen_pairs(block, matrix)
    scale = np.ones(block.shape[1]) if matrix == "cov" else 1.0 / block.std(axis = 0, ddof = 1)
    load = vec[:, 0] if n_pc == 1 else (vec[:, :n_pc] * lam[:n_pc]).sum(axis = 1) / lam[:n_pc].sum()
    w = np.clip(load * scale, 0, None)                  # keep the positive loadings only
    return cap_weights(w / w.sum()) if w.sum() > 0 else np.zeros(block.shape[1])


def book_maxvar(prices, matrix = "cov", n_pc = 1):
    # PC1 of the 252-day window, long only, capped, scaled to the volatility target
    return finish_book(maxvar_weights(recent_returns(prices, ESTIMATION_DAYS, STOCKS), matrix, n_pc), prices)

### <a id = "p1s4">Step 4.</a> Family B: residuals, the mean-reversion fit and the s-score [back to top](#top)

> `variance` keeps the factors that explain 55% of the variance; `noise` keeps those above the Marchenko-Pastur edge, the level pure noise would reach. Either way, 2 to 4 factors.

In [10]:
# Factors, residuals and s-scores of the stat-arb book (Week 8 notebook)
REG_DAYS = 60
REPLAY_DAYS = 30
B_MAX = 0.9672                 # largest slope b that still gives kappa = -252 ln b above 8.4


def choose_k(eigenvalues, k_rule = "variance", n_days = ESTIMATION_DAYS):
    # how many factors to keep, at least 2 and at most 4
    if k_rule == "variance":
        k = int(np.searchsorted(np.cumsum(eigenvalues) / eigenvalues.sum(), 0.55) + 1)
    else:
        k = int((eigenvalues > (1 + np.sqrt(len(eigenvalues) / n_days)) ** 2).sum())
    return int(np.clip(k, 2, 4))


def eigen_factors(block, k_rule = "variance"):
    # correlation PCA of a 252-day block of returns; returns the eigenportfolio weights Q = v / sigma and k
    lam, vec = eigen_pairs(block, "corr")
    k = choose_k(lam, k_rule, len(block))
    return vec[:, :k] / block.std(axis = 0, ddof = 1)[:, None], k


def ou_fit(stock_returns, factor_returns):
    # residuals, their running total and the mean-reversion fit, for every column of stock_returns at once
    design = np.column_stack([np.ones(len(factor_returns)), factor_returns])
    beta = np.linalg.lstsq(design, stock_returns, rcond = None)[0]
    residual = stock_returns - design @ beta
    X = residual.cumsum(axis = 0)
    x, y = X[:-1], X[1:]
    b = ((x - x.mean(axis = 0)) * (y - y.mean(axis = 0))).sum(axis = 0) / ((x - x.mean(axis = 0)) ** 2).sum(axis = 0)
    a = y.mean(axis = 0) - b * x.mean(axis = 0)
    zeta = y - (a + b * x)
    eligible = (b > 0) & (b < B_MAX)
    b_safe = np.where(eligible, b, 0.5)                    # placeholder slope for names that are not traded
    m = a / (1 - b_safe)
    sigma_eq = np.sqrt(zeta.var(axis = 0, ddof = 1) / (1 - b_safe ** 2))
    return m, sigma_eq, eligible


def s_scores(window, k_rule = "variance"):
    # the s-score of every stock on one day; window is the 252 x 12 array of returns ending that day
    Q, _ = eigen_factors(window, k_rule)
    m, sigma_eq, eligible = ou_fit(window[-REG_DAYS:], window[-REG_DAYS:] @ Q)
    if not eligible.any():
        return np.full(window.shape[1], np.nan)
    m_centred = m - m[eligible].mean()
    return np.where(eligible, -m_centred / sigma_eq, np.nan)

### <a id = "p1s5">Step 5.</a> Family B: replay the trading rule and turn it into weights [back to top](#top)

> Each call replays the open and close rule over the last 30 decision days, so the book gives the same answer however often it is called (R11).

In [11]:
def replay_step(state, s, open_long = -1.25, close_long = -0.50, open_under = 1.25, close_under = 0.75):
    # one day of the open and close rule, written for whole arrays
    no_signal = np.isnan(s)
    s = np.nan_to_num(s)
    state = np.where(state == 0, np.where(s < open_long, 1, np.where(s > open_under, -1, 0)), state)
    state = np.where((state == 1) & (s > close_long), 0, state)
    state = np.where((state == -1) & (s < close_under), 0, state)
    return np.where(no_signal, 0, state)


def statarb_states(prices, k_rule = "variance", replay = REPLAY_DAYS, open_long = -1.25, close_long = -0.50,
                   open_under = 1.25, close_under = 0.75):
    # +1 = overweight (oversold), -1 = underweight to zero (overbought), 0 = base
    R = recent_returns(prices, ESTIMATION_DAYS + replay, STOCKS)
    days = min(replay, len(R) - ESTIMATION_DAYS + 1)
    state = np.zeros(len(STOCKS))
    for end in range(len(R) - days + 1, len(R) + 1):
        state = replay_step(state, s_scores(R[end - ESTIMATION_DAYS : end], k_rule),
                            open_long, close_long, open_under, close_under)
    return state


def tilt_weights(state):
    # base 1/12, twice that when oversold, zero when overbought, then the cap of R7
    raw = np.where(state == 1, 2.0, np.where(state == -1, 0.0, 1.0))
    return cap_weights(raw / raw.sum(axis = -1, keepdims = True))


def book_statarb(prices, k_rule = "variance", replay = REPLAY_DAYS, vol_target = VOL_TARGET):
    # Avellaneda-Lee PCA stat-arb as a long-only tilt on equal weight
    return finish_book(tilt_weights(statarb_states(prices, k_rule, replay)), prices, target = vol_target)

### <a id = "p1s6">Step 6.</a> Four versions from two functions, and the yardsticks [back to top](#top)

> One loop over `FAMILIES` writes the four candidates. Everything else is fixed before any backtest. Equal weight and SPY are the yardsticks.

In [12]:
# Equal weight and SPY yardsticks, the two families and the four candidates (R5, R7)
def book_equal(prices):
    return finish_book(np.full(len(STOCKS), 1 / len(STOCKS)), prices, target = VOL_LIMIT)


def book_spy(prices):
    return finish_book(np.zeros(len(STOCKS)), prices, spy = 1.0, target = VOL_LIMIT)


FAMILIES = {"MaxVar": {"book": book_maxvar, "param": "matrix", "label": "matrix",
                       "grid": ["cov", "corr"], "ranks_on_signal": False},
            "StatArb": {"book": book_statarb, "param": "k_rule", "label": "k",
                        "grid": ["variance", "noise"], "ranks_on_signal": True}}

CANDIDATES, SETTING = {}, {}
for family, spec in FAMILIES.items():
    for value in spec["grid"]:
        name = f"{family} | {spec['label']} {value}"
        CANDIDATES[name] = partial(spec["book"], **{spec["param"]: value})
        SETTING[name] = (family, spec["param"], value)

YARDSTICKS = {"Equal weight": book_equal,
              "SPY": book_spy}

BOOKS = {**CANDIDATES, **YARDSTICKS}

COLORS = {"MaxVar | matrix cov": "#312E81", "MaxVar | matrix corr": "#b53a4a",
          "StatArb | k variance": "#1F8A8A", "StatArb | k noise": "#E07B39",
          "Equal weight": "#C9991A", "SPY": "#1B3556"}

list(BOOKS)

['MaxVar | matrix cov',
 'MaxVar | matrix corr',
 'StatArb | k variance',
 'StatArb | k noise',
 'Equal weight',
 'SPY']

### <a id = "p1s7">Step 7.</a> Check every candidate on one decision day [back to top](#top)

> Each book on 29 October 2026: the weights, the limits of R5 and the time of the call.

In [13]:
check_day = prices.loc[ : prices_frozen.index[-1]]
rows, seconds = {}, {}
for name, book in BOOKS.items():
    start = time.perf_counter()
    rows[name] = book(check_day)
    seconds[name] = time.perf_counter() - start
one_day =\
(
    pd
    .DataFrame(rows)
    .T
    .assign(cash = lambda f: 1 - f[TICKERS].sum(axis = 1),
            vol = lambda f: [ex_ante_vol(f.loc[n, TICKERS], check_day) for n in f.index],
            within_limits = lambda f: [check_limits(f.loc[n, TICKERS], check_day) for n in f.index],
            seconds = pd.Series(seconds))
)
one_day.round(4)

,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY,cash,vol,within_limits,seconds
MaxVar | matrix cov,0.162600,0.143400,0.158100,0.019700,0.079100,0.042900,0.000000,0.145900,0.000000,0.000000,0.184000,0.000000,0.000000,0.064300,0.195000,True,0.018700
MaxVar | matrix corr,0.159500,0.160100,0.176400,0.035600,0.124200,0.123300,0.000000,0.102600,0.000000,0.000000,0.118200,0.000000,0.000000,0.000000,0.182200,True,0.002400
StatArb | k variance,0.083300,0.166700,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.000000,0.083300,0.083300,0.000000,0.000000,0.119700,True,0.023400
StatArb | k noise,0.090900,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000,0.094000,True,0.014200
Equal weight,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.000000,0.000000,0.100000,True,0.000400
SPY,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.102000,True,0.001000


---

## <font color = "purple"> <a id = "part2">Part 2.</a> Backtest the candidates [back to top](#top)

### <a id = "p2s8">Step 8.</a> The desk engine and the scorecard [back to top](#top)

> The engine follows R6: decide at the close, earn the next day's return less 10 bp of the change in target weights, and hold cash for a day after a breach. Every call is timed. The scorecard is the one of R6, with USD 1,000,000 scaled from the window's first value in one run.

In [14]:
# Desk engine and scorecard (R6)
def run_desk(book, prices, first = FIRST_DECISION, last = WALK_LAST_DECISION):
    returns = prices[TICKERS].pct_change()
    decisions = prices.index[(prices.index >= first) & (prices.index <= last)]
    decisions = [d for d in decisions if prices.index.get_loc(d) + 1 < len(prices)]
    previous = pd.Series(0.0, index = TICKERS)
    rows, targets, breaches = [], {}, []
    for d in decisions:
        position = prices.index.get_loc(d)
        called = time.perf_counter()
        weights = book(prices.iloc[ : position + 1])
        seconds = time.perf_counter() - called
        if not check_limits(weights, prices.iloc[ : position + 1]):
            breaches.append(d)
            weights = pd.Series(0.0, index = TICKERS)
        weights = weights.reindex(TICKERS).fillna(0.0)
        day = prices.index[position + 1]
        rows.append({"date": day,
                     "decision": d,
                     "gross": weights @ returns.loc[day],
                     "turnover": (weights - previous).abs().sum(),
                     "invested": weights.sum(),
                     "seconds": seconds})
        targets[d] = weights
        previous = weights
    pnl =\
    (
        pd
        .DataFrame(rows)
        .set_index("date")
        .assign(net = lambda f: f["gross"] - COST * f["turnover"])
    )
    return pnl, pd.DataFrame(targets).T, breaches


def value_of(pnl):
    # value path from USD 1,000,000 at the close of the first decision day
    return CAPITAL * (1 + pnl["net"]).cumprod()


def scorecard(pnl, start, end):
    window = pnl.loc[start : end]
    daily = window["net"]
    value = value_of(pnl)
    v_start = CAPITAL if pnl.index.get_loc(window.index[0]) == 0 else value.iloc[pnl.index.get_loc(window.index[0]) - 1]
    path = pd.concat([pd.Series([v_start]), value.loc[start : end]]).reset_index(drop = True)
    return pd.Series({"Sharpe": daily.mean() / daily.std() * np.sqrt(252),
                      "CAGR": (path.iloc[-1] / v_start) ** (252 / len(daily)) - 1,
                      "vol": daily.std() * np.sqrt(252),
                      "max DD": (path / path.cummax() - 1).min(),
                      "turnover": window["turnover"].mean(),
                      "cost drag": COST * window["turnover"].mean() * 252,
                      "days scaled": int((window["invested"] < 1 - 1e-9).sum()),
                      "USD 1m becomes": CAPITAL * path.iloc[-1] / v_start})

### <a id = "p2s9">Step 9.</a> Walk forward, 3 January 2017 to 5 November 2026 [back to top](#top)

> One run per book over both windows. The breach count should be zero.

In [15]:
RUNS, TARGETS, BREACHES = {}, {}, {}
for name, book in BOOKS.items():
    start = time.perf_counter()
    RUNS[name], TARGETS[name], BREACHES[name] = run_desk(book, prices)
    print(f"{name:<24} {time.perf_counter() - start:7.1f} s   breaches: {len(BREACHES[name])}")

NET =\
(
    pd
    .DataFrame({name: pnl["net"] for name, pnl in RUNS.items()})
)
print("P&L days:", NET.index[0].date(), "to", NET.index[-1].date(), "|", len(NET))

MaxVar | matrix cov          6.2 s   breaches: 0
MaxVar | matrix corr         7.0 s   breaches: 0
StatArb | k variance        29.3 s   breaches: 0
StatArb | k noise           26.2 s   breaches: 0
Equal weight                 4.1 s   breaches: 0
SPY                          4.1 s   breaches: 0
P&L days: 2017-01-04 to 2026-11-06 | 2475


### <a id = "p2s10">Step 10.</a> Research window and holdout scorecards [back to top](#top)

> Run the holdout cells only after the selection rule is written. The candidate log keeps every version tried.

In [16]:
research =\
(
    pd
    .DataFrame({name: scorecard(pnl, RESEARCH_START, RESEARCH_END) for name, pnl in RUNS.items()})
    .T
)
print("Research window,", RESEARCH_START, "to", RESEARCH_END, "|", len(NET.loc[RESEARCH_START : RESEARCH_END]), "P&L days")
research.style.format({"Sharpe": "{:.4f}", "CAGR": "{:.2%}", "vol": "{:.2%}", "max DD": "{:.2%}",
                       "turnover": "{:.4f}", "cost drag": "{:.2%}", "days scaled": "{:,.0f}", "USD 1m becomes": "${:,.0f}"})

Research window, 2017-01-04 to 2024-12-31 | 2011 P&L days


,Sharpe,CAGR,vol,max DD,turnover,cost drag,days scaled,USD 1m becomes
MaxVar | matrix cov,1.3644,27.68%,19.28%,-28.41%,0.0103,0.26%,"1,367","$7,028,930"
MaxVar | matrix corr,1.1843,20.25%,16.77%,-23.08%,0.0104,0.26%,631,"$4,356,480"
StatArb | k variance,1.0991,18.53%,16.75%,-25.04%,0.1062,2.68%,670,"$3,883,550"
StatArb | k noise,1.1591,19.65%,16.68%,-24.67%,0.1055,2.66%,660,"$4,184,568"
Equal weight,1.2977,22.30%,16.58%,-23.16%,0.0033,0.08%,607,"$4,985,652"
SPY,0.8541,12.67%,15.36%,-25.66%,0.0026,0.07%,449,"$2,591,661"


In [17]:
def versus(table, against):
    return\
    (
        table[["Sharpe", "CAGR", "max DD", "USD 1m becomes"]]
        .sub(table.loc[against, ["Sharpe", "CAGR", "max DD", "USD 1m becomes"]])
        .drop(index = list(YARDSTICKS))
        .add_prefix("Δ ")
    )


(
    pd
    .concat({"vs equal weight": versus(research, "Equal weight"),
             "vs SPY": versus(research, "SPY")},
            axis = 1)
    .style
    .format("{:+.4f}")
    .format("{:+,.0f}", subset = [("vs equal weight", "Δ USD 1m becomes"), ("vs SPY", "Δ USD 1m becomes")])
    .format("{:+.2%}", subset = [(w, c) for w in ["vs equal weight", "vs SPY"] for c in ["Δ CAGR", "Δ max DD"]])
)

In [18]:
holdout =\
(
    pd
    .DataFrame({name: scorecard(pnl, HOLDOUT_START, HOLDOUT_END) for name, pnl in RUNS.items()})
    .T
)
print("Holdout,", HOLDOUT_START, "to", HOLDOUT_END, "|", len(NET.loc[HOLDOUT_START : HOLDOUT_END]), "P&L days")
holdout.style.format({"Sharpe": "{:.4f}", "CAGR": "{:.2%}", "vol": "{:.2%}", "max DD": "{:.2%}",
                      "turnover": "{:.4f}", "cost drag": "{:.2%}", "days scaled": "{:,.0f}", "USD 1m becomes": "${:,.0f}"})

Holdout, 2025-01-02 to 2026-11-06 | 464 P&L days


,Sharpe,CAGR,vol,max DD,turnover,cost drag,days scaled,USD 1m becomes
MaxVar | matrix cov,0.6668,11.85%,19.70%,-19.81%,0.0098,0.25%,289,"$1,228,977"
MaxVar | matrix corr,0.7693,13.44%,18.65%,-23.02%,0.0098,0.25%,186,"$1,261,447"
StatArb | k variance,1.6082,26.43%,15.31%,-16.58%,0.0972,2.45%,66,"$1,540,024"
StatArb | k noise,1.5126,24.49%,15.25%,-17.20%,0.0918,2.31%,66,"$1,496,895"
Equal weight,1.4052,22.10%,15.01%,-19.08%,0.0017,0.04%,66,"$1,444,417"
SPY,0.7149,10.68%,15.96%,-18.64%,0.0019,0.05%,66,"$1,205,338"


In [19]:
(
    pd
    .concat({"vs equal weight": versus(holdout, "Equal weight"),
             "vs SPY": versus(holdout, "SPY")},
            axis = 1)
    .style
    .format("{:+.4f}")
    .format("{:+,.0f}", subset = [("vs equal weight", "Δ USD 1m becomes"), ("vs SPY", "Δ USD 1m becomes")])
    .format("{:+.2%}", subset = [(w, c) for w in ["vs equal weight", "vs SPY"] for c in ["Δ CAGR", "Δ max DD"]])
)

In [20]:
# candidate log: every version tried, with both Sharpe ratios side by side
candidate_log =\
(
    pd
    .DataFrame({"research Sharpe": research["Sharpe"],
                "holdout Sharpe": holdout["Sharpe"],
                "research turnover": research["turnover"],
                "research days scaled": research["days scaled"]})
    .drop(index = list(YARDSTICKS))
    .sort_values("research Sharpe", ascending = False)
)
candidate_log.round(4)

,research Sharpe,holdout Sharpe,research turnover,research days scaled
MaxVar | matrix cov,1.364400,0.666800,0.010300,"1,367.000000"
MaxVar | matrix corr,1.184300,0.769300,0.010400,631.000000
StatArb | k noise,1.159100,1.512600,0.105500,660.000000
StatArb | k variance,1.099100,1.608200,0.106200,670.000000


### <a id = "p2s11">Step 11.</a> Charts [back to top](#top)

> Growth of USD 1,000,000, drawdown, headline numbers, and research against holdout Sharpe ratio.

In [21]:
# Growth of USD 1,000,000 over both windows (Week 8 notebook)
highlight = list(BOOKS)
palette = [COLORS[book] for book in highlight]

growth =\
(
    NET[highlight]
    .add(1)
    .cumprod()
    .mul(CAPITAL)
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "book", value_name = "value")
)
(
    ggplot(growth, aes(x = "date", y = "value", color = "book"))
    + geom_line(size = 0.9)
    + geom_vline(xintercept = pd.Timestamp(HOLDOUT_START), linetype = "dashed", color = "#8899AA")
    + scale_color_manual(values = palette, breaks = highlight)
    + scale_y_log10(format = "$,.0f")
    + labs(title = "Growth of USD 1,000,000, one walk-forward run over both windows",
           subtitle = "Decided at each close from 3 January 2017, 10 bp a trade. Dashed line: start of the holdout, 2 January 2025.",
           x = "", y = "Value (log scale)")
    + ggsize(950, 450)
)

In [22]:
# Drawdown from the running peak (Mini-Mock Review)
drawdown =\
(
    NET[highlight]
    .add(1)
    .cumprod()
    .pipe(lambda v: v / v.cummax() - 1)
    .rename_axis("date")
    .reset_index()
    .melt(id_vars = "date", var_name = "book", value_name = "drawdown")
)
(
    ggplot(drawdown, aes(x = "date", y = "drawdown", color = "book"))
    + geom_line(size = 0.6)
    + geom_vline(xintercept = pd.Timestamp(HOLDOUT_START), linetype = "dashed", color = "#8899AA")
    + scale_color_manual(values = palette, breaks = highlight)
    + scale_y_continuous(format = ".0%")
    + facet_wrap("book", ncol = 2)
    + labs(title = "Drawdown from the running peak",
           subtitle = "Dashed line: start of the holdout.",
           x = "", y = "Drawdown")
    + theme(legend_position = "none")
    + ggsize(950, 650)
)

In [23]:
headline =\
(
    pd
    .concat({"Research 2017-2024": research, "Holdout 2025-2026": holdout}, names = ["window", "book"])
    [["Sharpe", "CAGR", "max DD"]]
    .reset_index()
    .melt(id_vars = ["window", "book"], var_name = "measure", value_name = "value")
)
(
    ggplot(headline, aes(x = "book", y = "value", fill = "book"))
    + geom_bar(stat = "identity", position = "dodge")
    + geom_hline(yintercept = 0, color = "#8899AA")
    + facet_grid(x = "window", y = "measure", scales = "free_y")
    + scale_fill_manual(values = palette, breaks = highlight)
    + scale_x_discrete(limits = highlight)
    + coord_flip()
    + labs(title = "Sharpe ratio, CAGR and maximum drawdown in each window",
           subtitle = "The four versions of the two PCA families against equal weight of the twelve (gold) and SPY (navy).",
           x = "", y = "")
    + theme(legend_position = "none")
    + ggsize(1000, 800)
)

In [24]:
# Research Sharpe ratio against holdout Sharpe ratio
sharpe_pairs =\
(
    pd
    .DataFrame({"research": research["Sharpe"], "holdout": holdout["Sharpe"]})
    .rename_axis("book")
    .reset_index()
)
(
    ggplot(sharpe_pairs, aes(x = "research", y = "holdout", color = "book"))
    + geom_point(size = 6)
    + geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "#8899AA")
    + geom_hline(yintercept = 0, color = "#8899AA")
    + geom_vline(xintercept = 0, color = "#8899AA")
    + scale_color_manual(values = palette, breaks = highlight)
    + labs(title = "Sharpe ratio in the research window against the holdout",
           subtitle = "A book above the dashed line did better in the holdout than in the research window; one below it did worse.",
           x = "Research Sharpe ratio, 2017-2024", y = "Holdout Sharpe ratio, 2025-2026")
    + ggsize(900, 520)
)

### <a id = "p2s12">Step 12.</a> Is the research Sharpe ratio luck? t-statistic and deflated Sharpe ratio [back to top](#top)

> The R10 checks on the research window. The t-statistic asks how far the Sharpe ratio is from zero, allowing for lopsided and fat-tailed returns. The deflated Sharpe ratio (DSR) also allows for the $N$ versions tried. Bars: t above 3 and DSR above 0.95.

In [25]:
# t-statistic and deflated Sharpe ratio on the research window (R10)
EULER_GAMMA = 0.5772156649
N_TRIALS = len(CANDIDATES)


def sharpe_t_stat(daily):
    sr = daily.mean() / daily.std()
    g3, g4 = skew(daily), kurtosis(daily, fisher = False)
    se = np.sqrt((1 - g3 * sr + (g4 - 1) / 4 * sr ** 2) / (len(daily) - 1))
    return sr, sr / se, norm.cdf(-sr / se), g3, g4


def deflated_sharpe(daily, trial_sharpes, n_trials = N_TRIALS):
    sr, _, _, g3, g4 = sharpe_t_stat(daily)
    v = np.var(trial_sharpes, ddof = 1)
    sr_0 = np.sqrt(v) * ((1 - EULER_GAMMA) * norm.ppf(1 - 1 / n_trials)
                         + EULER_GAMMA * norm.ppf(1 - 1 / (n_trials * np.e)))
    return sr_0, norm.cdf((sr - sr_0) * np.sqrt(len(daily) - 1) / np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2))


research_net = NET.loc[RESEARCH_START : RESEARCH_END]
trial_sharpes = (research_net[list(CANDIDATES)].mean() / research_net[list(CANDIDATES)].std()).values

luck_rows = {}
for name in BOOKS:
    sr, t, p, g3, g4 = sharpe_t_stat(research_net[name])
    row = {"daily SR": sr, "annual SR": sr * np.sqrt(252), "skew": g3, "kurtosis": g4,
           "t": t, "p, one-sided": p, "Bonferroni p": min(1, N_TRIALS * p), "t > 3": t > 3}
    if name in CANDIDATES:
        row["SR_0"], row["DSR"] = deflated_sharpe(research_net[name], trial_sharpes)
    luck_rows[name] = row

luck =\
(
    pd
    .DataFrame(luck_rows)
    .T
    .sort_values("annual SR", ascending = False)
)
print("N =", N_TRIALS, "| V =", f"{np.var(trial_sharpes, ddof = 1):.3e}", "| T =", len(research_net))
luck

N = 4 | V = 5.172e-05 | T = 2011


,daily SR,annual SR,skew,kurtosis,t,"p, one-sided",Bonferroni p,t > 3,SR_0,DSR
MaxVar | matrix cov,0.085948,1.364378,-0.340049,5.642474,3.782470,0.000078,0.000311,True,0.007567,0.999719
Equal weight,0.081748,1.297706,-0.594717,6.579636,3.563230,0.000183,0.000733,True,NaN,NaN
MaxVar | matrix corr,0.074604,1.184309,-0.560725,6.421793,3.265103,0.000547,0.002188,True,0.007567,0.998327
StatArb | k noise,0.073019,1.159148,-0.558953,6.459121,3.197697,0.000693,0.002771,True,0.007567,0.997924
StatArb | k variance,0.069238,1.099114,-0.563421,6.448979,3.035778,0.001200,0.004798,True,0.007567,0.996575
SPY,0.053802,0.854077,-0.672457,8.121556,2.363742,0.009046,0.036183,False,NaN,NaN


### <a id = "p2s13">Step 13.</a> Placebo [back to top](#top)

> The R10 placebo, for the book that ranks stocks on a signal (stat-arb only). Each stock's signal is shifted in time at random, 500 draws with seed 627, and the p-value is the share of placebo Sharpe ratios at or above the actual one. The second cell checks the fast version against the book.

In [26]:
# Placebo: shift each stock's signal in time and rebuild the book (R10, Mini-Mock Review)
PLACEBO_BOOKS = [name for name, (family, _, _) in SETTING.items() if FAMILIES[family]["ranks_on_signal"]]


def s_score_panel(prices, decisions, k_rule = "variance"):
    R = prices[STOCKS].pct_change().values
    scores = []
    for d in decisions:
        end = prices.index.get_loc(d)
        scores.append(s_scores(R[end - ESTIMATION_DAYS + 1 : end + 1], k_rule))
    return np.array(scores)


def replay_panel(scores, replay = REPLAY_DAYS):
    # the state on every decision day, replaying the last `replay` days from flat, on whole arrays
    n = len(scores)
    state = np.zeros(scores.shape)
    for lag in range(replay - 1, -1, -1):
        rows = np.arange(n) - lag
        valid = rows >= 0
        stepped = replay_step(state, scores[np.clip(rows, 0, None)])
        state = np.where(valid[:, None], stepped, state)
    return state


def raw_sharpe(states, next_returns):
    weights = tilt_weights(states)
    turnover = np.abs(np.diff(weights, axis = 0, prepend = 0.0)).sum(axis = 1)
    net = (weights * next_returns).sum(axis = 1) - COST * turnover
    return net.mean() / net.std(ddof = 1) * np.sqrt(252)


research_decisions = list(RUNS["Equal weight"].loc[RESEARCH_START : RESEARCH_END, "decision"])
next_returns = prices[STOCKS].pct_change().loc[RESEARCH_START : RESEARCH_END].values
print("books through the placebo:", PLACEBO_BOOKS)
print("research decisions:", len(research_decisions), research_decisions[0].date(), "to", research_decisions[-1].date())

books through the placebo: ['StatArb | k variance', 'StatArb | k noise']
research decisions: 2011 2017-01-03 to 2024-12-30


In [27]:
# check that the panel shortcut gives the same states as the book, on 40 days spread over the window
sample = np.linspace(0, len(research_decisions) - 1, 40).astype(int)
for name in PLACEBO_BOOKS:
    k_rule = SETTING[name][2]
    base_states = replay_panel(s_score_panel(prices, research_decisions, k_rule))
    print(f"{name}: panel states equal the book's states:",
          all((statarb_states(prices.loc[ : research_decisions[i]], k_rule) == base_states[i]).all() for i in sample))

StatArb | k variance: panel states equal the book's states: True
StatArb | k noise: panel states equal the book's states: True


In [28]:
PLACEBO_DRAWS = 500

placebo_rows, placebo_draws = {}, {}
for name in PLACEBO_BOOKS:
    scores = s_score_panel(prices, research_decisions, SETTING[name][2])
    actual = raw_sharpe(replay_panel(scores), next_returns)
    rng = np.random.default_rng(seed = SEED)
    n = len(scores)
    draws = []
    for _ in range(PLACEBO_DRAWS):
        shifts = rng.integers(252, n - 252, size = len(STOCKS))
        rolled_scores = np.column_stack([np.roll(scores[:, i], shifts[i]) for i in range(len(STOCKS))])
        draws.append(raw_sharpe(replay_panel(rolled_scores), next_returns))
    draws = np.array(draws)
    placebo_draws[name] = draws
    placebo_rows[name] = {"actual raw Sharpe": actual,
                          "placebo mean": draws.mean(),
                          "placebo 95th pct": np.percentile(draws, 95),
                          "p-value": (1 + (draws >= actual).sum()) / (PLACEBO_DRAWS + 1)}

placebo =\
(
    pd
    .DataFrame(placebo_rows)
    .T
)
placebo.round(4)

,actual raw Sharpe,placebo mean,placebo 95th pct,p-value
StatArb | k variance,1.045600,1.056700,1.163900,0.572900
StatArb | k noise,1.096500,1.056500,1.163600,0.287400


In [29]:
best = placebo["actual raw Sharpe"].idxmax()
(
    ggplot(pd.DataFrame({"placebo Sharpe": placebo_draws[best]}), aes(x = "placebo Sharpe"))
    + geom_histogram(bins = 40, fill = "#8899AA", color = "white")
    + geom_vline(xintercept = placebo.loc[best, "actual raw Sharpe"], color = "#b53a4a", size = 1.2)
    + labs(title = f"Placebo of R10 for {best}: {PLACEBO_DRAWS} time-shifted signals on the research window",
           subtitle = f"Red line: the actual signal. p-value = {placebo.loc[best, 'p-value']:.4f}. Raw tilt book, no scaler, 10 bp on turnover.",
           x = "Sharpe ratio of the placebo book", y = "count")
    + ggsize(900, 380)
)

### <a id = "p2s14">Step 14.</a> The ten-day lens of the holdout [back to top](#top)

> Blocks of ten holdout days (R6), dropping the incomplete last block. The last cell scales them to the four unseen days: mean times 0.4 and sd times $\sqrt{0.4}$, shown as the mean plus and minus two sds.

In [30]:
# Ten-day blocks of the holdout (R6)
holdout_net = NET.loc[HOLDOUT_START : HOLDOUT_END]
n_blocks = len(holdout_net) // 10

blocks =\
(
    holdout_net
    .iloc[ : n_blocks * 10]
    .add(1)
    .groupby(np.arange(n_blocks * 10) // 10)
    .prod()
    .sub(1)
)

lens =\
(
    pd
    .DataFrame({"mean": blocks.mean(),
                "sd": blocks.std(),
                "worst": blocks.min(),
                "best": blocks.max(),
                "beat SPY": blocks.gt(blocks["SPY"], axis = 0).mean()})
    .sort_values("mean", ascending = False)
)
print("holdout P&L days:", len(holdout_net), "| blocks:", n_blocks, "| days over:", len(holdout_net) - n_blocks * 10)
lens.style.format({"mean": "{:.2%}", "sd": "{:.2%}", "worst": "{:.2%}", "best": "{:.2%}", "beat SPY": "{:.2%}"})

holdout P&L days: 464 | blocks: 46 | days over: 4


,mean,sd,worst,best,beat SPY
StatArb | k variance,0.99%,2.82%,-4.57%,10.67%,71.74%
StatArb | k noise,0.96%,2.79%,-4.82%,9.39%,67.39%
Equal weight,0.86%,2.76%,-5.25%,9.45%,71.74%
MaxVar | matrix corr,0.59%,3.92%,-8.82%,13.13%,50.00%
MaxVar | matrix cov,0.56%,4.26%,-9.68%,14.94%,58.70%
SPY,0.49%,2.59%,-5.28%,9.89%,0.00%


In [31]:
lens_frame =\
(
    blocks[highlight]
    .reset_index(drop = True)
    .melt(var_name = "book", value_name = "ten-day return")
)
(
    ggplot(lens_frame, aes(x = "book", y = "ten-day return", fill = "book"))
    + geom_boxplot(alpha = 0.8)
    + geom_hline(yintercept = 0, color = "#8899AA")
    + scale_fill_manual(values = palette, breaks = highlight)
    + scale_x_discrete(limits = highlight)
    + scale_y_continuous(format = ".1%")
    + labs(title = f"Ten-day returns in the holdout, {n_blocks} blocks from 2 January 2025",
           subtitle = "The live window is one draw from a distribution like this one.",
           x = "", y = "Ten-day return")
    + theme(legend_position = "none")
    + ggsize(1000, 420)
)

In [32]:
# Range for 17 to 20 November: ten-day mean times 0.4 and ten-day sd times the square root of 0.4
UNSEEN_DAYS = 4
share = UNSEEN_DAYS / 10

unseen_range =\
(
    pd
    .DataFrame({"ten-day mean": lens["mean"],
                "ten-day sd": lens["sd"],
                "four-day mean": lens["mean"] * share,
                "four-day sd": lens["sd"] * np.sqrt(share)})
    .assign(low = lambda f: f["four-day mean"] - 2 * f["four-day sd"],
            high = lambda f: f["four-day mean"] + 2 * f["four-day sd"])
)
unseen_range.style.format("{:.2%}")

,ten-day mean,ten-day sd,four-day mean,four-day sd,low,high
StatArb | k variance,0.99%,2.82%,0.40%,1.79%,-3.17%,3.97%
StatArb | k noise,0.96%,2.79%,0.39%,1.77%,-3.14%,3.92%
Equal weight,0.86%,2.76%,0.34%,1.75%,-3.15%,3.84%
MaxVar | matrix corr,0.59%,3.92%,0.24%,2.48%,-4.72%,5.19%
MaxVar | matrix cov,0.56%,4.26%,0.22%,2.69%,-5.16%,5.61%
SPY,0.49%,2.59%,0.20%,1.64%,-3.08%,3.48%


---

## <font color = "purple"> <a id = "live">The days seen before the deadline, 9 to 16 November</a> [back to top](#top)

> Each book is formed from cash at the 6 November close and scored on the six P&L days of 9 to 16 November. Six days estimate nothing; they are one draw from the distribution of Step 14.

### <a id = "lives15">Step 15.</a> What the books did on the six days [back to top](#top)

In [33]:
live_runs, live_pnl = {}, {}
for name, book in BOOKS.items():
    pnl, _, breaches = run_desk(book, prices, first = LIVE_FIRST_DECISION, last = prices.index[-2])
    live_runs[name] = pnl
    live_pnl[name] = pnl.loc[LIVE_START : , "net"]
    if breaches:
        print("limit breaches:", name, [f"{d:%Y-%m-%d}" for d in breaches])
live = pd.DataFrame(live_pnl)
print("live window:", f"{live.index[0]:%Y-%m-%d}", "to", f"{live.index[-1]:%Y-%m-%d}", "|", len(live), "P&L days")

live_value = CAPITAL * (1 + live).cumprod()
live_summary =\
(
    pd
    .DataFrame({"return": live_value.iloc[-1] / CAPITAL - 1,
                "USD 1m becomes": live_value.iloc[-1],
                "cost of the first day": [CAPITAL * COST * live_runs[name]["turnover"].iloc[0] for name in live.columns]})
)
display(live.style.format("{:.2%}"))
live_summary.style.format({"return": "{:.2%}", "USD 1m becomes": "${:,.2f}", "cost of the first day": "${:,.2f}"})

live window: 2026-11-09 to 2026-11-16 | 6 P&L days


,MaxVar | matrix cov,MaxVar | matrix corr,StatArb | k variance,StatArb | k noise,Equal weight,SPY
date,,,,,,
2026-11-09 00:00:00,2.02%,2.09%,1.61%,2.11%,1.70%,2.27%
2026-11-10 00:00:00,-1.15%,-1.02%,-0.94%,-0.95%,-0.77%,-0.37%
2026-11-11 00:00:00,0.41%,0.34%,1.27%,0.96%,0.53%,1.32%
2026-11-12 00:00:00,-0.38%,-0.23%,0.21%,0.04%,0.08%,-0.35%
2026-11-13 00:00:00,-0.03%,-0.02%,0.30%,0.45%,0.41%,-0.36%
2026-11-16 00:00:00,-2.90%,-3.00%,-2.05%,-2.14%,-2.12%,-1.91%


,return,USD 1m becomes,cost of the first day
MaxVar | matrix cov,-2.08%,"$979,182.05",$897.71
MaxVar | matrix corr,-1.89%,"$981,129.37","$1,000.00"
StatArb | k variance,0.35%,"$1,003,549.98","$1,000.00"
StatArb | k noise,0.41%,"$1,004,088.24","$1,000.00"
Equal weight,-0.22%,"$997,843.81","$1,000.00"
SPY,0.54%,"$1,005,354.61","$1,000.00"


### <a id = "lives16">Step 16.</a> Weights at every close from 6 to 16 November [back to top](#top)

> Each book is called on the prices up to that close and checked against R5.

In [34]:
live_closes = prices.loc["2026-11-06" : ].index
live_weights = {name: pd.DataFrame({d: book(prices.loc[ : d]) for d in live_closes}).T.rename_axis("close")
                for name, book in CANDIDATES.items()}

for name, table in live_weights.items():
    print(name)
    display(table.assign(cash = 1 - table.sum(axis = 1)).round(4))

print("every candidate within the limits of R5 at every close from 6 to 16 November:",
      all(check_limits(table.loc[d], prices.loc[ : d]) for table in live_weights.values() for d in live_closes))

MaxVar | matrix cov


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY,cash
close,,,,,,,,,,,,,,
2026-11-06,0.160000,0.135800,0.146900,0.024300,0.084200,0.048200,0.000000,0.130100,0.000000,0.000000,0.168200,0.000000,0.000000,0.102300
2026-11-09,0.149200,0.132800,0.142700,0.025200,0.081200,0.049000,0.000000,0.132000,0.000000,0.000000,0.164700,0.000000,0.000000,0.123200
2026-11-10,0.150500,0.131000,0.140500,0.025100,0.081200,0.048600,0.000000,0.131900,0.000000,0.000000,0.163000,0.000000,0.000000,0.128100
2026-11-11,0.148400,0.130300,0.141300,0.025400,0.080700,0.048900,0.000000,0.133200,0.000000,0.000000,0.164300,0.000000,0.000000,0.127400
2026-11-12,0.148500,0.130100,0.141300,0.026000,0.080800,0.048900,0.000000,0.132900,0.000000,0.000000,0.164400,0.000000,0.000000,0.127000
2026-11-13,0.149300,0.130100,0.141100,0.025900,0.081000,0.049200,0.000000,0.133800,0.000000,0.000000,0.163700,0.000000,0.000000,0.125900
2026-11-16,0.144900,0.126900,0.136700,0.025600,0.077400,0.047300,0.000000,0.131500,0.000000,0.000000,0.159500,0.000000,0.000000,0.150200


MaxVar | matrix corr


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY,cash
close,,,,,,,,,,,,,,
2026-11-06,0.139500,0.158200,0.178200,0.053400,0.118900,0.134300,0.000000,0.096300,0.000000,0.000000,0.121300,0.000000,0.000000,0.000000
2026-11-09,0.130500,0.156300,0.175200,0.057100,0.116500,0.136200,0.000000,0.093700,0.000000,0.000000,0.118300,0.000000,0.000000,0.016300
2026-11-10,0.128500,0.156000,0.174200,0.057200,0.116300,0.136300,0.000000,0.093500,0.000000,0.000000,0.118200,0.000000,0.000000,0.019700
2026-11-11,0.129300,0.156200,0.174500,0.056200,0.115900,0.135800,0.000000,0.093400,0.000000,0.000000,0.118500,0.000000,0.000000,0.020200
2026-11-12,0.127900,0.155900,0.174900,0.059400,0.116400,0.136200,0.000000,0.093000,0.000000,0.000000,0.118600,0.000000,0.000000,0.017600
2026-11-13,0.128100,0.156600,0.175800,0.059300,0.116600,0.136800,0.000000,0.093400,0.000000,0.000000,0.118100,0.000000,0.000000,0.015300
2026-11-16,0.123200,0.150300,0.168800,0.058600,0.111600,0.132800,0.000000,0.090300,0.000000,0.000000,0.114400,0.000000,0.000000,0.050000


StatArb | k variance


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY,cash
close,,,,,,,,,,,,,,
2026-11-06,0.100000,0.200000,0.000000,0.200000,0.000000,0.000000,0.100000,0.100000,0.100000,0.000000,0.100000,0.100000,0.000000,0.000000
2026-11-09,0.100000,0.200000,0.000000,0.200000,0.000000,0.000000,0.100000,0.100000,0.100000,0.000000,0.100000,0.100000,0.000000,0.000000
2026-11-10,0.090900,0.181800,0.000000,0.181800,0.000000,0.000000,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,0.000000
2026-11-11,0.090900,0.181800,0.000000,0.090900,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000
2026-11-12,0.090900,0.181800,0.000000,0.090900,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000
2026-11-13,0.090900,0.181800,0.090900,0.090900,0.090900,0.000000,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000
2026-11-16,0.090900,0.181800,0.090900,0.090900,0.090900,0.000000,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000


StatArb | k noise


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY,cash
close,,,,,,,,,,,,,,
2026-11-06,0.090900,0.181800,0.000000,0.181800,0.000000,0.000000,0.090900,0.181800,0.090900,0.000000,0.090900,0.090900,0.000000,0.000000
2026-11-09,0.100000,0.200000,0.000000,0.200000,0.000000,0.000000,0.100000,0.100000,0.100000,0.000000,0.100000,0.100000,0.000000,0.000000
2026-11-10,0.083300,0.166700,0.000000,0.166700,0.083300,0.000000,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.000000,0.000000
2026-11-11,0.090900,0.090900,0.000000,0.181800,0.090900,0.000000,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,-0.000000
2026-11-12,0.083300,0.083300,0.000000,0.166700,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.000000,0.000000
2026-11-13,0.090900,0.090900,0.000000,0.181800,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.090900,0.000000,0.000000,-0.000000
2026-11-16,0.166700,0.083300,0.000000,0.166700,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.083300,0.000000,0.000000,0.000000


every candidate within the limits of R5 at every close from 6 to 16 November: True


### <a id = "lives17">Step 17.</a> How the four versions relate, and the names held at the 16 November close [back to top](#top)

> Book 2 must correlate below 0.90 with Book 1 in the holdout, or it holds cash on the live window.

In [35]:
holdout_corr = NET.loc[HOLDOUT_START : HOLDOUT_END, list(CANDIDATES)].corr()
display(holdout_corr.style.format("{:.3f}"))

held =\
(
    pd
    .DataFrame({name: table.loc[live_closes[-1]].loc[lambda s: s > 1e-6] for name, table in live_weights.items()})
    .sort_index()
)
print("names held at the", f"{live_closes[-1]:%d %B %Y}", "close")
held.style.format("{:.1%}", na_rep = "")

,MaxVar | matrix cov,MaxVar | matrix corr,StatArb | k variance,StatArb | k noise
MaxVar | matrix cov,1.000,0.964,0.863,0.849
MaxVar | matrix corr,0.964,1.000,0.911,0.898
StatArb | k variance,0.863,0.911,1.000,0.977
StatArb | k noise,0.849,0.898,0.977,1.000


names held at the 16 November 2026 close


,MaxVar | matrix cov,MaxVar | matrix corr,StatArb | k variance,StatArb | k noise
AAPL,2.6%,5.9%,9.1%,16.7%
AMZN,13.7%,16.9%,9.1%,
CAT,13.1%,9.0%,,8.3%
GOOGL,12.7%,15.0%,18.2%,8.3%
JNJ,,,9.1%,8.3%
JPM,4.7%,13.3%,,8.3%
KO,,,9.1%,
META,16.0%,11.4%,9.1%,8.3%
MSFT,7.7%,11.2%,9.1%,8.3%
NVDA,14.5%,12.3%,9.1%,16.7%


In [36]:
# Heat map of the holdout correlations of the four versions
corr_long =(
    holdout_corr
    .rename_axis("book x")
    .reset_index()
    .melt(id_vars = "book x", var_name = "book y", value_name = "correlation")
    .assign(label = lambda f: f["correlation"].round(2))
)
(
    ggplot(corr_long, aes(x = "book x", y = "book y", fill = "correlation"))
    + geom_tile(color = "white")
    + geom_text(aes(label = "label"), size = 8)
    + scale_fill_gradient2(low = "#2166AC", mid = "#F7F7F7", high = "#B2182B", midpoint = 0, limits = [-1, 1])
    + theme(axis_text_x = element_text(angle = 25, hjust = 1))
    + labs(title = "Correlation of the daily returns of the four versions in the holdout",
           subtitle = "Book 2 must correlate below 0.90 with Book 1.",
           x = "", y = "")
    + ggsize(800, 600)
)

---

# Self-critique of the books [back to top](#top)

> Adapted from Cao Jie

### <a id = "p4s18">Step 18.</a> Market regimes [back to top](#top)

> Sharpe ratio, drawdown and return in five periods, with the dates fixed before the results. *Days scaled*: days the volatility limit cut the book.

In [37]:
# Sharpe ratio, drawdown, return and days scaled by regime
REGIMES = {"2017-2019": (RESEARCH_START, "2019-12-31"),
           "COVID crash": ("2020-02-19", "2020-04-30"),
           "2022 selloff": ("2022-01-03", "2022-12-30"),
           "2023-2024": ("2023-01-03", RESEARCH_END),
           "Holdout": (HOLDOUT_START, HOLDOUT_END)}

regime_rows = {}
for regime, (start, end) in REGIMES.items():
    for name, pnl in RUNS.items():
        card = scorecard(pnl, start, end)
        regime_rows[(regime, name)] = {"Sharpe": card["Sharpe"],
                                       "max DD": card["max DD"],
                                       "total return": card["USD 1m becomes"] / CAPITAL - 1,
                                       "days": len(pnl.loc[start : end]),
                                       "days scaled": card["days scaled"]}

regime_table =(
    pd
    .DataFrame(regime_rows)
    .T
    .rename_axis(["regime", "book"])
)
assert np.allclose(regime_table.loc["Holdout", "Sharpe"], holdout["Sharpe"].reindex(regime_table.loc["Holdout"].index))
regime_table.style.format({"Sharpe": "{:.4f}", "max DD": "{:.2%}", "total return": "{:.2%}", "days": "{:,.0f}", "days scaled": "{:,.0f}"})

### <a id = "p4s19">Step 19.</a> Costs and the breakeven cost [back to top](#top)

> Research-window returns re-priced at 0, 10, 25 and 50 bp. The breakeven cost is the cost per trade at which the average daily return falls to zero.

In [38]:
# Sharpe ratio at several costs, and the breakeven cost
cost_rows = {}
for name, pnl in RUNS.items():
    window = pnl.loc[RESEARCH_START : RESEARCH_END]
    row = {"cost drag at 10 bp": COST * window["turnover"].mean() * 252,
           "breakeven cost (bp)": window["gross"].mean() / window["turnover"].mean() * 10_000}
    for bp in [0, 10, 25, 50]:
        net = window["gross"] - bp / 10_000 * window["turnover"]
        row[f"Sharpe at {bp} bp"] = net.mean() / net.std() * np.sqrt(252)
    cost_rows[name] = row

cost_table =(
    pd
    .DataFrame(cost_rows)
    .T
)
assert np.allclose(cost_table["Sharpe at 10 bp"], research["Sharpe"].reindex(cost_table.index))
cost_table.style.format({"cost drag at 10 bp": "{:.2%}", "breakeven cost (bp)": "{:,.1f}", "Sharpe at 0 bp": "{:.4f}",
                         "Sharpe at 10 bp": "{:.4f}", "Sharpe at 25 bp": "{:.4f}", "Sharpe at 50 bp": "{:.4f}"})

,cost drag at 10 bp,breakeven cost (bp),Sharpe at 0 bp,Sharpe at 10 bp,Sharpe at 25 bp,Sharpe at 50 bp
MaxVar | matrix cov,0.26%,"1,019.4",1.3778,1.3644,1.3442,1.3104
MaxVar | matrix corr,0.26%,768.6,1.1999,1.1843,1.1608,1.1217
StatArb | k variance,2.68%,78.8,1.2588,1.0991,0.8594,0.4598
StatArb | k noise,2.66%,82.7,1.3184,1.1591,0.9200,0.5212
Equal weight,0.08%,"2,584.1",1.3027,1.2977,1.2901,1.2775
SPY,0.07%,"2,022.9",0.8583,0.8541,0.8477,0.8372


### <a id = "p4s20">Step 20.</a> Lookahead probes [back to top](#top)

> Each book is called again on the prices up to four decision days. The cell checks the weights, that the P&L is booked on the next trading day, and that day's return.

In [39]:
# Check the weights, the timing and the return of each book on four decision days
probe_days = [pd.Timestamp(d) for d in ["2019-12-31", "2020-03-31", "2024-12-31", "2026-11-05"]]
daily_returns = prices[TICKERS].pct_change()

probe_rows = []
for name, book in BOOKS.items():
    for day in probe_days:
        position = prices.index.get_loc(day)
        again = book(prices.iloc[ : position + 1]).reindex(TICKERS).fillna(0.0)
        used = TARGETS[name].loc[day]
        pnl_day = prices.index[position + 1]
        booked = RUNS[name].loc[pnl_day]
        probe_rows.append({"book": name,
                           "decision": f"{day:%Y-%m-%d}",
                           "P&L day": f"{pnl_day:%Y-%m-%d}",
                           "weights match": bool(np.allclose(again, used, atol = 1e-12)),
                           "next trading day": bool(booked["decision"] == day),
                           "gross is next day's return": bool(np.isclose(booked["gross"], used @ daily_returns.loc[pnl_day], atol = 1e-12))})

probes = pd.DataFrame(probe_rows)
assert probes[["weights match", "next trading day", "gross is next day's return"]].all().all()
print("every probe passes:", len(probes), "probes,", probes["book"].nunique(), "books,", len(probe_days), "decision days")
probes

every probe passes: 24 probes, 6 books, 4 decision days


,book,decision,P&L day,weights match,next trading day,gross is next day's return
0,MaxVar | matrix cov,2019-12-31,2020-01-02,True,True,True
1,MaxVar | matrix cov,2020-03-31,2020-04-01,True,True,True
2,MaxVar | matrix cov,2024-12-31,2025-01-02,True,True,True
3,MaxVar | matrix cov,2026-11-05,2026-11-06,True,True,True
4,MaxVar | matrix corr,2019-12-31,2020-01-02,True,True,True
5,MaxVar | matrix corr,2020-03-31,2020-04-01,True,True,True
6,MaxVar | matrix corr,2024-12-31,2025-01-02,True,True,True
7,MaxVar | matrix corr,2026-11-05,2026-11-06,True,True,True
8,StatArb | k variance,2019-12-31,2020-01-02,True,True,True
9,StatArb | k variance,2020-03-31,2020-04-01,True,True,True


### <a id = "p4s21">Step 21.</a> Timing and limits on every decision day [back to top](#top)

> The slowest call (R11: under 30 seconds) and the limit breaches over all decision days, read from the Step 9 run.

In [40]:
# Call time and limit breaches of each book, from the walk-forward run
timing =(
    pd
    .DataFrame({name: {"decisions": len(pnl),
                       "mean call (ms)": pnl["seconds"].mean() * 1000,
                       "slowest call (ms)": pnl["seconds"].max() * 1000,
                       "breaches": len(BREACHES[name])}
                for name, pnl in RUNS.items()})
    .T
)
assert timing["slowest call (ms)"].max() < CALL_LIMIT_SECONDS * 1000, "a call took longer than the 30 seconds of R11"
print("slowest call of any book:", f"{timing['slowest call (ms)'].max() / 1000:.2f}", "s | decisions that breached a limit:", int(timing["breaches"].sum()))
timing.style.format({"decisions": "{:,.0f}", "mean call (ms)": "{:,.1f}", "slowest call (ms)": "{:,.1f}", "breaches": "{:,.0f}"})

slowest call of any book: 0.34 s | decisions that breached a limit: 0


,decisions,mean call (ms),slowest call (ms),breaches
MaxVar | matrix cov,"2,475",0.7,6.4,0
MaxVar | matrix corr,"2,475",0.9,54.7,0
StatArb | k variance,"2,475",10.0,335.4,0
StatArb | k noise,"2,475",8.9,99.7,0
Equal weight,"2,475",0.3,3.9,0
SPY,"2,475",0.3,0.6,0


### <a id = "p4s22">Step 22.</a> Survivorship and multiple testing [back to top](#top)

> Excess Sharpe ratio over equal weight and over SPY, and the placebo p-value (stat-arb only). The twelve names were chosen in 2026, so survivorship bias stays.

In [41]:
# Excess Sharpe ratio over the yardsticks, and the placebo p-value, of each candidate
candidates = list(CANDIDATES)
survivorship =(
    pd
    .DataFrame({"research Sharpe": research.loc[candidates, "Sharpe"],
                "over equal weight": research.loc[candidates, "Sharpe"] - research.loc["Equal weight", "Sharpe"],
                "over SPY": research.loc[candidates, "Sharpe"] - research.loc["SPY", "Sharpe"],
                "placebo p-value": placebo["p-value"].reindex(candidates)})
)
print("versions with t > 3:", int(luck.loc[candidates, "t"].astype(float).gt(3).sum()), "of", N_TRIALS,
      "| versions with DSR > 0.95:", int(luck.loc[candidates, "DSR"].astype(float).gt(0.95).sum()), "of", N_TRIALS)
survivorship.style.format({"research Sharpe": "{:.4f}", "over equal weight": "{:+.4f}", "over SPY": "{:+.4f}", "placebo p-value": "{:.4f}"}, na_rep = "")

versions with t > 3: 4 of 4 | versions with DSR > 0.95: 4 of 4


,research Sharpe,over equal weight,over SPY,placebo p-value
MaxVar | matrix cov,1.3644,+0.0667,+0.5103,
MaxVar | matrix corr,1.1843,-0.1134,+0.3302,
StatArb | k variance,1.0991,-0.1986,+0.2450,0.5729
StatArb | k noise,1.1591,-0.1386,+0.3051,0.2874


## Design choices

The guideline only gives us adjusted closes, so both books start from the same raw material: the daily returns of the 12 stocks, with SPY left out of the PCA. I built both books from one PCA, which splits the stocks' moves into independent directions ordered by how much of the variance each explains. That way the two books differ only in what I do with the result. The max-variance book takes PC1, the direction in which the stocks move together the most. It keeps the positive weights because the desk is long only, caps each stock at 25% and scales the book to the volatility limit. The stat-arb book of Avellaneda and Lee (2010) removes the first few directions from each stock's return and trades the leftover. When a stock's leftover has fallen further than usual (an s-score below −1.25), the stock is oversold. The paper shorts the overbought names, which the desk does not allow. So I start from equal weight, double the oversold names and set the overbought names to zero.

Each family has one open setting. For the max-variance book it is the matrix that is decomposed (covariance or correlation). For the stat-arb book it is how many factors to keep (55% of the variance, or everything above the noise edge). That gives 4 versions, which keeps the penalty of the deflated Sharpe ratio small. Everything else was fixed before the backtest: the 252-day and 60-day windows, the thresholds (−1.25, −0.50, +1.25, +0.75), the 30-day replay and the 19.5% volatility target. These come from the paper and the Week 8 notebook, and I did not tune them here.

## Critique of the max-variance book

This book spends the whole risk budget on the stocks that move the most, and the volatility scaler pays for it. On 29 October the covariance version put 16.3% in NVDA, 18.4% in META, 15.8% in AMZN and 14.3% in GOOGL, and nothing in JNJ, XOM, WMT or KO. The 20% limit cut it on 1,367 of the 2,011 research days (68%), against 607 days (30%) for equal weight. In 2022 it was cut on all 251 days. Even so, its research volatility was 19.3% against 16.6% for equal weight, and its maximum drawdown was −28.4% against −23.2%. The scaler also looks back 63 days, so it cannot help in a sudden fall. In the COVID crash (19 February to 30 April 2020) every book lost between 14% and 17%, and the covariance book lost 14.3%, no better than equal weight's 14.2%.

The research edge is small, and I do not trust where it comes from. Its Sharpe ratio of 1.364 is only 0.067 above equal weight's 1.298. It is below equal weight in these periods: 2017 to 2019, COVID, 2022, 2023 to 2024. So whatever lead it has over the full window sits in the days those periods leave out, mostly May 2020 to December 2021. It also did worst in 2022, losing 22.5% against 15.2% for equal weight. The correlation version is calmer (cut on 631 days, drawdown −23.1%), but its research Sharpe ratio of 1.184 is 0.113 below equal weight.

The guidelien chose these 12 names in 2026, and the largest weights of this book go to the loudest of them, which are the names that grew the most. The book is partly buying the stocks that were always going to stay in the list. I have no placebo for it, because it ranks nothing, so its edge rests on the t-statistic (3.78) and the deflated Sharpe ratio alone. In the holdout its Sharpe ratio fell to 0.67, below SPY's 0.71 and less than half of equal weight's 1.41. On the six days seen it lost 2.08%, against 0.22% for equal weight.

## Critique of the stat-arb book

The stat-arb book does not beat holding the 12 stocks equally. Its research Sharpe ratio was 1.099 (variance) and 1.159 (noise), against 1.298 for equal weight. It trailed equal weight in all four research periods, except that the noise version matched it in 2022 (−0.68 for both).

Costs explain most of this. The tilt turns over 0.106 of the book per day, ten times the max-variance book's 0.010, and that costs 2.7% a year. With no cost at all, the Sharpe ratios are 1.259 and 1.318, against 1.303 for equal weight. So before costs the signal adds nothing for the variance version and only 0.016 for the noise version. The breakeven cost is the cost per trade at which the average return falls to zero. It is 79 and 83 bp for the stat-arb versions, against 1,019 and 769 bp for the max-variance versions. At 50 bp the stat-arb Sharpe ratios fall to 0.46 and 0.52. The desk charges 10 bp, so the books survive, but only about eight times below their breakeven.

The placebo says the timing of the signal may not matter at all. I shifted each stock's s-score in time by a random number of days, 500 times, and rebuilt the book each time. The actual raw Sharpe ratio of the variance version was 1.046, below the placebo average of 1.057 (p = 0.57). The noise version's was 1.097, above the placebo average but below the placebo 95th percentile of 1.164 (p = 0.29). Neither book clears the usual 5% bar. So I cannot rule out that the stat-arb books earn what they earn from holding these 12 names and not from the signal. The holdout looks better: Sharpe ratios of 1.61 and 1.51 against 1.41 for equal weight, and gains of 0.35% and 0.41% on the six days seen against a 0.22% loss for equal weight. But 464 days is too short to separate that from luck, as the next section shows.

## Critique of the parameter grid

I did not search over settings: each family has two values of one setting. The same worries apply anyway. All four versions pass a t-statistic of 3 (3.04 to 3.78) and have deflated Sharpe ratios between 0.997 and 1.000 for $N$ = 4, with a hurdle $SR_0$ of 0.0076 a day, about 0.12 a year. That looks safe, but equal weight also has a t-statistic of 3.56. Passing the bar says the 12 names did well from 2017 to 2024, not that PCA added anything. The deflation is also only as strict as the count. It counts 4 versions, and these are not independent: the two versions in a family share all their code and differ in one setting. If I pool our versions with the other members', the count rises to $N$ = 11, the hurdle rises to about 0.70 a year and the deflated Sharpe ratio of the covariance book falls to about 0.968.

The differences between versions are also small compared with how precisely a Sharpe ratio can be measured. An annualised Sharpe ratio $\widehat{SR}$ estimated over $Y$ years has a standard error of roughly

$$\text{se}\left(\widehat{SR}\right) \approx \sqrt{\frac{1 + \widehat{SR}^2/2}{Y}}$$

Over the 8-year research window, with a Sharpe ratio near 1.2, that is about $\sqrt{1.7/8} \approx 0.46$. My four versions ranged from 1.10 to 1.36, a spread of 0.27, well inside that error. So the order among them is not reliable. The holdout supports this, because it reversed both orders inside the families. The correlation version beat the covariance version (0.77 against 0.67), and the variance version beat the noise version (1.61 against 1.51). The holdout is only 1.84 years long, so its own standard error is about 0.9, and its numbers cannot settle the question either.

## Choice of my Book 1 candidates

In the research window, the covariance version led the max-variance books (1.364 against 1.184) and the noise version led the stat-arb books (1.159 against 1.099). So I put forward MaxVar | matrix cov and StatArb | k noise, the research leader of each family. I made this choice by the research order. But I have seen the holdout table, so for these two books the holdout is a second look and not a clean test. Part 4 reports this as a lookahead risk. The holdout put the two versions I set aside ahead of the two I picked within each family.

## My strategy functions
Each function is self-contained and fixed to one version, so any of them can be copied into the strategy file on its own.

In [42]:
def book_pca_maxvar_cov(prices):
    import numpy as np
    import pandas as pd

    STOCKS = ["NVDA", "GOOGL", "AMZN", "AAPL", "MSFT", "JPM", "JNJ", "CAT", "XOM", "WMT", "META", "KO"]
    TICKERS = STOCKS + ["SPY"]
    STOCK_CAP = 0.25
    VOL_TARGET = 0.195
    RISK_DAYS = 63
    ESTIMATION_DAYS = 252

    def recent_returns(days, tickers):
        closes = prices[tickers].values[-(days + 1):]
        return closes[1:] / closes[:-1] - 1

    def cap_weights(w, cap = STOCK_CAP):
        w = np.asarray(w, dtype = float).copy()
        for _ in range(len(w)):
            over = w > cap + 1e-12
            if not over.any():
                break
            excess = (w[over] - cap).sum()
            w[over] = cap
            room = (w > 0) & (w < cap)
            if room.sum() == 0 or w[room].sum() == 0:
                break
            w[room] = w[room] + excess * w[room] / w[room].sum()
        return w

    def vol_scaler(weights, returns_13):
        sigma_63 = np.cov(returns_13[-RISK_DAYS:].T)
        vol = np.sqrt(252 * weights @ sigma_63 @ weights)
        return weights * min(1.0, VOL_TARGET / vol) if vol > 0 else weights

    block = recent_returns(ESTIMATION_DAYS, STOCKS)
    lam, vec = np.linalg.eigh(np.cov(block.T))
    order = np.argsort(lam)[::-1]
    lam, vec = lam[order], vec[:, order]
    vec = vec * np.sign(vec.sum(axis = 0))

    w = np.clip(vec[:, 0], 0, None)
    stock_weights = cap_weights(w / w.sum()) if w.sum() > 0 else np.zeros(block.shape[1])

    weights = vol_scaler(np.append(stock_weights, 0.0), recent_returns(RISK_DAYS, TICKERS))
    return pd.Series(weights, index = TICKERS)


def book_pca_maxvar_corr(prices):
    import numpy as np
    import pandas as pd

    STOCKS = ["NVDA", "GOOGL", "AMZN", "AAPL", "MSFT", "JPM", "JNJ", "CAT", "XOM", "WMT", "META", "KO"]
    TICKERS = STOCKS + ["SPY"]
    STOCK_CAP = 0.25
    VOL_TARGET = 0.195
    RISK_DAYS = 63
    ESTIMATION_DAYS = 252

    def recent_returns(days, tickers):
        closes = prices[tickers].values[-(days + 1):]
        return closes[1:] / closes[:-1] - 1

    def cap_weights(w, cap = STOCK_CAP):
        w = np.asarray(w, dtype = float).copy()
        for _ in range(len(w)):
            over = w > cap + 1e-12
            if not over.any():
                break
            excess = (w[over] - cap).sum()
            w[over] = cap
            room = (w > 0) & (w < cap)
            if room.sum() == 0 or w[room].sum() == 0:
                break
            w[room] = w[room] + excess * w[room] / w[room].sum()
        return w

    def vol_scaler(weights, returns_13):
        sigma_63 = np.cov(returns_13[-RISK_DAYS:].T)
        vol = np.sqrt(252 * weights @ sigma_63 @ weights)
        return weights * min(1.0, VOL_TARGET / vol) if vol > 0 else weights

    block = recent_returns(ESTIMATION_DAYS, STOCKS)
    lam, vec = np.linalg.eigh(np.corrcoef(block.T))
    order = np.argsort(lam)[::-1]
    lam, vec = lam[order], vec[:, order]
    vec = vec * np.sign(vec.sum(axis = 0))

    w = np.clip(vec[:, 0] / block.std(axis = 0, ddof = 1), 0, None)
    stock_weights = cap_weights(w / w.sum()) if w.sum() > 0 else np.zeros(block.shape[1])

    weights = vol_scaler(np.append(stock_weights, 0.0), recent_returns(RISK_DAYS, TICKERS))
    return pd.Series(weights, index = TICKERS)


def book_pca_statarb_variance(prices):
    import numpy as np
    import pandas as pd

    STOCKS = ["NVDA", "GOOGL", "AMZN", "AAPL", "MSFT", "JPM", "JNJ", "CAT", "XOM", "WMT", "META", "KO"]
    TICKERS = STOCKS + ["SPY"]
    STOCK_CAP = 0.25
    VOL_TARGET = 0.195
    RISK_DAYS = 63
    ESTIMATION_DAYS = 252
    REG_DAYS = 60
    REPLAY_DAYS = 30
    B_MAX = 0.9672

    def recent_returns(days, tickers):
        closes = prices[tickers].values[-(days + 1):]
        return closes[1:] / closes[:-1] - 1

    def cap_weights(w, cap = STOCK_CAP):
        w = np.asarray(w, dtype = float).copy()
        for _ in range(len(w)):
            over = w > cap + 1e-12
            if not over.any():
                break
            excess = (w[over] - cap).sum()
            w[over] = cap
            room = (w > 0) & (w < cap)
            if room.sum() == 0 or w[room].sum() == 0:
                break
            w[room] = w[room] + excess * w[room] / w[room].sum()
        return w

    def vol_scaler(weights, returns_13):
        sigma_63 = np.cov(returns_13[-RISK_DAYS:].T)
        vol = np.sqrt(252 * weights @ sigma_63 @ weights)
        return weights * min(1.0, VOL_TARGET / vol) if vol > 0 else weights

    def eigen_factors(block):
        lam, vec = np.linalg.eigh(np.corrcoef(block.T))
        order = np.argsort(lam)[::-1]
        lam, vec = lam[order], vec[:, order]
        vec = vec * np.sign(vec.sum(axis = 0))
        k = int(np.searchsorted(np.cumsum(lam) / lam.sum(), 0.55) + 1)
        k = int(np.clip(k, 2, 4))
        return vec[:, :k] / block.std(axis = 0, ddof = 1)[:, None]

    def s_scores(window):
        Q = eigen_factors(window)
        stock_returns, factor_returns = window[-REG_DAYS:], window[-REG_DAYS:] @ Q
        design = np.column_stack([np.ones(len(factor_returns)), factor_returns])
        beta = np.linalg.lstsq(design, stock_returns, rcond = None)[0]
        residual = stock_returns - design @ beta
        X = residual.cumsum(axis = 0)
        x, y = X[:-1], X[1:]
        b = ((x - x.mean(axis = 0)) * (y - y.mean(axis = 0))).sum(axis = 0) / ((x - x.mean(axis = 0)) ** 2).sum(axis = 0)
        a = y.mean(axis = 0) - b * x.mean(axis = 0)
        zeta = y - (a + b * x)
        eligible = (b > 0) & (b < B_MAX)
        if not eligible.any():
            return np.full(window.shape[1], np.nan)
        b_safe = np.where(eligible, b, 0.5)
        m = a / (1 - b_safe)
        sigma_eq = np.sqrt(zeta.var(axis = 0, ddof = 1) / (1 - b_safe ** 2))
        return np.where(eligible, -(m - m[eligible].mean()) / sigma_eq, np.nan)

    def replay_step(state, s):
        no_signal = np.isnan(s)
        s = np.nan_to_num(s)
        state = np.where(state == 0, np.where(s < -1.25, 1, np.where(s > 1.25, -1, 0)), state)
        state = np.where((state == 1) & (s > -0.50), 0, state)
        state = np.where((state == -1) & (s < 0.75), 0, state)
        return np.where(no_signal, 0, state)

    R = recent_returns(ESTIMATION_DAYS + REPLAY_DAYS, STOCKS)
    days = min(REPLAY_DAYS, len(R) - ESTIMATION_DAYS + 1)
    state = np.zeros(len(STOCKS))
    for end in range(len(R) - days + 1, len(R) + 1):
        state = replay_step(state, s_scores(R[end - ESTIMATION_DAYS : end]))

    raw = np.where(state == 1, 2.0, np.where(state == -1, 0.0, 1.0))
    stock_weights = cap_weights(raw / raw.sum())

    weights = vol_scaler(np.append(stock_weights, 0.0), recent_returns(RISK_DAYS, TICKERS))
    return pd.Series(weights, index = TICKERS)


def book_pca_statarb_noise(prices):
    import numpy as np
    import pandas as pd

    STOCKS = ["NVDA", "GOOGL", "AMZN", "AAPL", "MSFT", "JPM", "JNJ", "CAT", "XOM", "WMT", "META", "KO"]
    TICKERS = STOCKS + ["SPY"]
    STOCK_CAP = 0.25
    VOL_TARGET = 0.195
    RISK_DAYS = 63
    ESTIMATION_DAYS = 252
    REG_DAYS = 60
    REPLAY_DAYS = 30
    B_MAX = 0.9672

    def recent_returns(days, tickers):
        closes = prices[tickers].values[-(days + 1):]
        return closes[1:] / closes[:-1] - 1

    def cap_weights(w, cap = STOCK_CAP):
        w = np.asarray(w, dtype = float).copy()
        for _ in range(len(w)):
            over = w > cap + 1e-12
            if not over.any():
                break
            excess = (w[over] - cap).sum()
            w[over] = cap
            room = (w > 0) & (w < cap)
            if room.sum() == 0 or w[room].sum() == 0:
                break
            w[room] = w[room] + excess * w[room] / w[room].sum()
        return w

    def vol_scaler(weights, returns_13):
        sigma_63 = np.cov(returns_13[-RISK_DAYS:].T)
        vol = np.sqrt(252 * weights @ sigma_63 @ weights)
        return weights * min(1.0, VOL_TARGET / vol) if vol > 0 else weights

    def eigen_factors(block):
        lam, vec = np.linalg.eigh(np.corrcoef(block.T))
        order = np.argsort(lam)[::-1]
        lam, vec = lam[order], vec[:, order]
        vec = vec * np.sign(vec.sum(axis = 0))
        k = int((lam > (1 + np.sqrt(len(lam) / len(block))) ** 2).sum())
        k = int(np.clip(k, 2, 4))
        return vec[:, :k] / block.std(axis = 0, ddof = 1)[:, None]

    def s_scores(window):
        Q = eigen_factors(window)
        stock_returns, factor_returns = window[-REG_DAYS:], window[-REG_DAYS:] @ Q
        design = np.column_stack([np.ones(len(factor_returns)), factor_returns])
        beta = np.linalg.lstsq(design, stock_returns, rcond = None)[0]
        residual = stock_returns - design @ beta
        X = residual.cumsum(axis = 0)
        x, y = X[:-1], X[1:]
        b = ((x - x.mean(axis = 0)) * (y - y.mean(axis = 0))).sum(axis = 0) / ((x - x.mean(axis = 0)) ** 2).sum(axis = 0)
        a = y.mean(axis = 0) - b * x.mean(axis = 0)
        zeta = y - (a + b * x)
        eligible = (b > 0) & (b < B_MAX)
        if not eligible.any():
            return np.full(window.shape[1], np.nan)
        b_safe = np.where(eligible, b, 0.5)
        m = a / (1 - b_safe)
        sigma_eq = np.sqrt(zeta.var(axis = 0, ddof = 1) / (1 - b_safe ** 2))
        return np.where(eligible, -(m - m[eligible].mean()) / sigma_eq, np.nan)

    def replay_step(state, s):
        no_signal = np.isnan(s)
        s = np.nan_to_num(s)
        state = np.where(state == 0, np.where(s < -1.25, 1, np.where(s > 1.25, -1, 0)), state)
        state = np.where((state == 1) & (s > -0.50), 0, state)
        state = np.where((state == -1) & (s < 0.75), 0, state)
        return np.where(no_signal, 0, state)

    R = recent_returns(ESTIMATION_DAYS + REPLAY_DAYS, STOCKS)
    days = min(REPLAY_DAYS, len(R) - ESTIMATION_DAYS + 1)
    state = np.zeros(len(STOCKS))
    for end in range(len(R) - days + 1, len(R) + 1):
        state = replay_step(state, s_scores(R[end - ESTIMATION_DAYS : end]))

    raw = np.where(state == 1, 2.0, np.where(state == -1, 0.0, 1.0))
    stock_weights = cap_weights(raw / raw.sum())

    weights = vol_scaler(np.append(stock_weights, 0.0), recent_returns(RISK_DAYS, TICKERS))
    return pd.Series(weights, index = TICKERS)

In [43]:
# Check that each self-contained function gives the weights of its notebook version, on the closes of 6 to 16 November
# and on earlier days, including the first days of 2017, then print the two put-forward books' weights at every close
check_days = prices.loc["2026-11-06":].index.union(pd.DatetimeIndex(["2017-01-03", "2017-02-14", "2019-12-31", "2020-03-31", "2024-12-31", "2026-10-29"]))

pairs =\
{
    "MaxVar | matrix cov": book_pca_maxvar_cov,
    "MaxVar | matrix corr": book_pca_maxvar_corr,
    "StatArb | k variance": book_pca_statarb_variance,
    "StatArb | k noise": book_pca_statarb_noise,
}

for name, book in pairs.items():
    largest_gap = max((book(prices.loc[:d]) - CANDIDATES[name](prices.loc[:d])).abs().max() for d in check_days)
    within_limits = all(check_limits(book(prices.loc[:d]), prices.loc[:d]) for d in check_days)
    print(f"{name:<22} largest gap against the notebook version: {largest_gap:.2e} | within the limits of R5: {within_limits}")

close_days = prices.loc["2026-11-06":].index
weights_maxvar = pd.DataFrame({f"{d:%Y-%m-%d}": book_pca_maxvar_cov(prices.loc[:d]) for d in close_days}).T
weights_statarb = pd.DataFrame({f"{d:%Y-%m-%d}": book_pca_statarb_noise(prices.loc[:d]) for d in close_days}).T

print("MaxVar | matrix cov, weights at each close")
display(weights_maxvar.round(6))
print("StatArb | k noise, weights at each close")
display(weights_statarb.round(6))

MaxVar | matrix cov    largest gap against the notebook version: 0.00e+00 | within the limits of R5: True
MaxVar | matrix corr   largest gap against the notebook version: 2.78e-17 | within the limits of R5: True
StatArb | k variance   largest gap against the notebook version: 0.00e+00 | within the limits of R5: True
StatArb | k noise      largest gap against the notebook version: 0.00e+00 | within the limits of R5: True
MaxVar | matrix cov, weights at each close


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY
2026-11-06,0.160048,0.135760,0.146875,0.024331,0.084169,0.048221,0.000000,0.130095,0.000000,0.000000,0.168210,0.000000,0.000000
2026-11-09,0.149168,0.132807,0.142652,0.025217,0.081217,0.049000,0.000000,0.131972,0.000000,0.000000,0.164739,0.000000,0.000000
2026-11-10,0.150482,0.131020,0.140515,0.025115,0.081238,0.048634,0.000000,0.131909,0.000000,0.000000,0.162977,0.000000,0.000000
2026-11-11,0.148391,0.130315,0.141290,0.025427,0.080688,0.048898,0.000000,0.133241,0.000000,0.000000,0.164311,0.000000,0.000000
2026-11-12,0.148487,0.130129,0.141337,0.026029,0.080848,0.048866,0.000000,0.132900,0.000000,0.000000,0.164391,0.000000,0.000000
2026-11-13,0.149258,0.130109,0.141139,0.025877,0.081024,0.049234,0.000000,0.133797,0.000000,0.000000,0.163704,0.000000,0.000000
2026-11-16,0.144909,0.126917,0.136675,0.025605,0.077373,0.047338,0.000000,0.131490,0.000000,0.000000,0.159532,0.000000,0.000000


StatArb | k noise, weights at each close


,NVDA,GOOGL,AMZN,AAPL,MSFT,JPM,JNJ,CAT,XOM,WMT,META,KO,SPY
2026-11-06,0.090909,0.181818,0.000000,0.181818,0.000000,0.000000,0.090909,0.181818,0.090909,0.000000,0.090909,0.090909,0.000000
2026-11-09,0.100000,0.200000,0.000000,0.200000,0.000000,0.000000,0.100000,0.100000,0.100000,0.000000,0.100000,0.100000,0.000000
2026-11-10,0.083333,0.166667,0.000000,0.166667,0.083333,0.000000,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.000000
2026-11-11,0.090909,0.090909,0.000000,0.181818,0.090909,0.000000,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.000000
2026-11-12,0.083333,0.083333,0.000000,0.166667,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.000000
2026-11-13,0.090909,0.090909,0.000000,0.181818,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.000000,0.000000
2026-11-16,0.166667,0.083333,0.000000,0.166667,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.000000,0.000000
